<a href="https://colab.research.google.com/github/woomook0524/punctuation-paper/blob/main/0119(K%3D2_%EC%8B%A4%ED%97%982_%EC%A0%84%EC%B2%B4_dataset_third).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 라이브러리 설치
!pip install -q transformers accelerate scikit-learn

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

# 결과 재현성을 위한 시드 고정
def set_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
#cell 2

from huggingface_hub import login
login()

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()

# 구두점 ID 매핑
PUNCT_MAP = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1]
}
EOS_ID = tokenizer.eos_token_id
print(f"Punctuation IDs: {PUNCT_MAP}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Punctuation IDs: {'COMMA': 11, 'PERIOD': 13, 'QMARK': 30}


In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

text = "Unbelievably, I really like cat."
print("tokens:", tok.tokenize(text))
print("ids   :", tok(text, add_special_tokens=False)["input_ids"])

tokens: ['Un', 'belie', 'vably', ',', 'ĠI', 'Ġreally', 'Ġlike', 'Ġcat', '.']
ids   : [1844, 32898, 89234, 11, 358, 2216, 1093, 8415, 13]


In [ ]:
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 사용자 제공 경로 및 로드 코드
BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"
VAL_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")

assert os.path.exists(VAL_Y_PATH), f"Not found: {VAL_Y_PATH}"

print(f"Loading file: {VAL_Y_PATH}")
with open(VAL_Y_PATH, "r", encoding="utf-8") as f:
    val_y_list = [line.strip() for line in f if line.strip()]

with open(TEST_Y_PATH, "r", encoding="utf-8") as f:
    test_y_list = [line.strip() for line in f if line.strip()]

print("num val samples:", len(val_y_list))
print("example:", val_y_list[0])


Mounted at /content/drive
Loading file: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_validation.Y.txt
num val samples: 1501
example: Last year I showed these two slides so that demonstrate that the arctic ice cap, which for most of the last three million years has been the size of the lower 48 states, has shrunk by 40 percent.


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. 설정 ---
# 물음표(QMARK) 샘플 확보를 위해 전체 구간 사용
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 4  # K=4 Tokens (Strict)

# 구두점 토큰 ID
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

def parse_sentence_to_boundaries(text: str):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        # 패턴: (내용)(구두점)(닫는 따옴표/괄호/대괄호 등 0개 이상)
        m = re.match(r'^(.*?)([,.?])([\"\'\)\]\}]*)$', tok)

        if m:
            base, punct_char = m.group(1), m.group(2)

            if punct_char == ',':
                label = "COMMA"
            elif punct_char == '.':
                label = "PERIOD"
            elif punct_char == '?':
                label = "QMARK"

            word = base
        else:
            label = "O"
            word = tok

        # 단어쪽의 따옴표/괄호 제거(필요 최소한)
        word = re.sub(r'[\"\'\(\)\[\]\{\}]', '', word)

        # 따옴표만 있는 토큰(") 같은 건 boundary로 취급하지 않음
        if word:
            boundaries.append({"word": word, "label": label})
            continue

        # 케이스 2: 단어 없이 구두점만 있는 토큰 (e.g., '."', ',"', '?"')
        # 이 구두점은 직전 단어의 라벨로 귀속
        if m and boundaries:
            # 직전 라벨이 이미 구두점이면 덮어쓸지/유지할지 정책 필요
            # 보통은 마지막에 나온 구두점이 더 강하니 덮어씀을 추천
            boundaries[-1]["label"] = label

        # 케이스 3: 따옴표만 있는 토큰(") 등은 그냥 무시
        # (m도 아니고 word도 비면 여기로 옴)

    return boundaries

# 헬퍼 함수: Chunking을 이용한 확률 합 계산
def get_joint_score_optimized(prefix_ids, target_ids):
    """
    prefix_ids: 현재까지의 문맥 토큰들
    target_ids: 미래 K개의 토큰들 (lookahead_tokens)
    """
    if not target_ids: return 0.0
    full_input = torch.tensor([prefix_ids + target_ids], device=device)
    with torch.no_grad():
        out = model(full_input)

    start_pos = len(prefix_ids) - 1
    # 미래 K개 토큰 위치에 해당하는 Logits 추출
    rel_logits = out.logits[0, start_pos : start_pos + len(target_ids), :]
    log_probs = torch.log_softmax(rel_logits, dim=-1)
    t_ids_tensor = torch.tensor(target_ids, device=device)

    # 해당 토큰들의 확률 합(Joint Probability) 반환
    return log_probs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. 설정 ---
# 물음표(QMARK) 샘플 확보를 위해 전체 구간 사용
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 2  # K=2 Tokens (Strict)

# 구두점 토큰 ID
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

# --- 2. 데이터 수집 (Strict Logic 적용) ---
print(f"Step 1: Collecting scores for K={K_VALUE} (Strict K Tokens, Full Future)...")
raw_data = []

# EOS 토큰 확인 (Padding용)
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(dataset_subset)):
    # 텍스트 파싱
    boundaries = parse_sentence_to_boundaries(y_true)

    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word = item['word']
        gold_label = item['label']
        current_words.append(word)

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety Check] tokenizer가 EOS를 붙이는지 확인 (Llama는 보통 안 붙임)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
             # 만약 끝에 EOS가 있다면 제거 (모델이 문장이 끝났다고 착각하지 않게)
            h_ids = h_ids[:-1]

        # Lookahead Token 준비 (Full Future & Strict Padding)
        # 미래의 모든 단어를 가져와서 토큰화
        full_future_words = [b['word'] for b in boundaries[i+1:]] # 전체 미래 단어
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            # 미래 텍스트가 아예 없으면 EOS로만 채움
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            # 전체 인코딩
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)

            # 앞에서 K개 자르기
            lookahead_tokens = full_future_ids[:K_VALUE]

            # [Strict Padding] K개보다 적으면 EOS로 채우기
            if len(lookahead_tokens) < K_VALUE:
                padding_len = K_VALUE - len(lookahead_tokens)
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * padding_len)

        # S0: No Punctuation Score
        s0 = get_joint_score_optimized(h_ids, lookahead_tokens)

        # Cost (Current Word Logits)
        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        # Batch Candidates (Batching + Chunking)
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        entry = {"gold": gold_label}
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            entry[f"{pname}_cost"] = base_logprobs[pid].item()

            # Gain 추출
            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            entry[f"{pname}_gain"] = p_joint_score - s0

        raw_data.append(entry)

        # Teacher Forcing Update
        if gold_label != "O":
            punct_char = "," if gold_label == "COMMA" else ("." if gold_label == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

df_scores = pd.DataFrame(raw_data)

# --- 3. 그리드 서치 (Punctuation-Focused Metric) ---
print("\nStep 2: Searching for optimal parameters...")
alpha_range = np.arange(0.1, 0.95, 0.05)
threshold_range = np.arange(-3.0, 2.0, 0.25)

best_score = -1
best_params = {}
golds = df_scores["gold"].values

# 'O'를 제외한 라벨들만 평가 대상으로 설정
EVAL_LABELS = ["COMMA", "PERIOD", "QMARK"]

for alpha in alpha_range:
    for thresh in threshold_range:
        preds = []
        for _, row in df_scores.iterrows():
            best_p, max_s = "O", float("-inf")
            for pname in ["COMMA", "PERIOD", "QMARK"]:
                # 논문 언급대로 Convex Combination 사용 (실험적 튜닝)
                score = (alpha * row[f"{pname}_cost"]) + ((1 - alpha) * row[f"{pname}_gain"])
                if score > max_s:
                    max_s, best_p = score, pname
            if max_s <= thresh: best_p = "O"
            preds.append(best_p)

        # 'O' 제외 Macro F1을 사용하여 구두점 성능 최적화
        # 'average="macro"'로 설정하여 소수 클래스(QMARK 등)의 성능도 중요하게 반영
        current_score = f1_score(golds, preds, average="macro", labels=EVAL_LABELS, zero_division=0)

        if current_score > best_score:
            best_score, best_params = current_score, {"alpha": alpha, "threshold": thresh}

print(f"\n=== K={K_VALUE} (Strict Token) Optimization Results ===")
print(f"Target Metric: Macro F1 (excluding 'O')")
print(f"Best Macro F1: {best_score:.4f}")
print(f"Optimal ALPHA: {best_params['alpha']:.2f}")
print(f"Optimal THRESHOLD: {best_params['threshold']:.2f}")

Step 1: Collecting scores for K=2 (Strict K Tokens, Full Future)...


100%|██████████| 1501/1501 [29:13<00:00,  1.17s/it]



Step 2: Searching for optimal parameters...

=== K=2 (Strict Token) Optimization Results ===
Target Metric: Macro F1 (excluding 'O')
Best Macro F1: 0.8730
Optimal ALPHA: 0.55
Optimal THRESHOLD: -1.00


미래의 2 tokens lookahead 가정 시

최적의 alpha와 threshold 계산

alpha_range = np.arange(0.1, 0.95, 0.05)

threshold_range = np.arange(-3.0, 2.0, 0.25)


In [ ]:
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

# --- 1. 파라미터 설정 (K=4 최적값) ---
ALPHA_K4 = 0.55       # Calibration 결과
THRESHOLD_K4 = -1.00  # Calibration 결과
K_VALUE = 2           # Strict K=2

# 출력 순서
LABELS_ORDER = ["O", "COMMA", "PERIOD", "QMARK"]

# 구두점 토큰 ID
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("?", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())


# --- 2. 평가 루프 (2-Pass Optimization 적용) ---
print(f"Starting Final Evaluation (K={K_VALUE}, Strict Token Logic, 2-Pass Optimization)")
print(f"Params: Alpha={ALPHA_K4}, Threshold={THRESHOLD_K4}")
print("Metric: Words/second")

all_golds = []
all_preds = []
total_processed_words = 0
start_time = time.time()
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(test_y_list)):
    boundaries = parse_sentence_to_boundaries(y_true)
    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word, gold_label = item['word'], item['label']
        current_words.append(word)
        total_processed_words += 1

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID: h_ids = h_ids[:-1]

        # [Strict K=4 Lookahead]
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE]
            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        # === 2-Pass Optimization 핵심 ===
        # Pass 1: History + Lookahead를 한 번에 넣어서 (Cost)와 (S0)를 동시에 구함
        # -------------------------------------------------------------------------
        # 입력 구조: [h_1, ..., h_n, t_1, ..., t_k]
        full_input_ids = h_ids + lookahead_tokens
        full_input_tensor = torch.tensor([full_input_ids], device=device)

        with torch.no_grad():
            out_1 = model(full_input_tensor)

        # (1) Cost 추출: History의 마지막 토큰(h_n) 위치에서의 Logits
        # h_n의 위치는 index `len(h_ids) - 1`
        # 이 Logits은 "다음 토큰 예측" 확률이므로, 여기서 구두점 확률(Cost)을 가져옴
        base_logits = out_1.logits[0, len(h_ids) - 1, :]
        base_logprobs = torch.log_softmax(base_logits, dim=-1)

        # (2) S0 추출: History 끝부터 Lookahead 끝까지의 확률 (Joint Probability)
        # Logits 범위: `len(h_ids) - 1` (첫 Lookahead 예측) ~ `len(full_input) - 2` (마지막 Lookahead 예측)
        start_pos = len(h_ids) - 1
        target_len = len(lookahead_tokens)

        # Lookahead 부분의 Logits 추출
        # out_1.logits[0, start_pos : start_pos + target_len] -> [K, Vocab]
        s0_logits = out_1.logits[0, start_pos : start_pos + target_len, :]
        s0_logprobs = torch.log_softmax(s0_logits, dim=-1)
        s0_target_ids = torch.tensor(lookahead_tokens, device=device)

        # 정답 토큰(Lookahead tokens)의 확률 합산
        s0 = s0_logprobs.gather(1, s0_target_ids.unsqueeze(1)).squeeze(1).sum().item()

        # Pass 2: Batch Forward (Gain 계산)
        # -------------------------------------------------------------------------
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)

        with torch.no_grad():
            batch_out = model(batch_tensor)

        best_p, max_s = "O", float('-inf')
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            # Cost는 아까 Pass 1에서 구한 것 사용
            cost = base_logprobs[pid].item()

            # Gain 추출
            # batch 입력 구조: [h_1...h_n, PUNCT, t_1...t_k]
            # PUNCT 위치는 len(h_ids). t_1 예측은 len(h_ids) 위치의 Logit에서 나옴.
            start_pos_batch = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos_batch : start_pos_batch + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            gain = p_joint_score - s0

            score = (ALPHA_K4 * cost) + ((1 - ALPHA_K4) * gain)

            if score > THRESHOLD_K4 and score > max_s:
                max_s, best_p = score, pname

        all_golds.append(gold_label)
        all_preds.append(best_p)

        if best_p != "O":
            punct_char = "," if best_p == "COMMA" else ("." if best_p == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

# --- 3. 결과 리포트 ---
end_time = time.time()
elapsed = end_time - start_time
wps = total_processed_words / elapsed

print(f"\n[Final Results | K={K_VALUE} Strict 2-Pass Optimized]")
print(f"Inference Speed: {wps:.2f} words/s")
print(f"Total Processed Words: {total_processed_words}")
print(f"Total Execution Time: {elapsed:.2f}s")
print("-" * 60)
print(classification_report(all_golds, all_preds, labels=LABELS_ORDER, zero_division=0, digits=3))

print("\nConfusion Matrix")
cm = confusion_matrix(all_golds, all_preds, labels=LABELS_ORDER)
df_cm = pd.DataFrame(cm, index=[f"True_{l}" for l in LABELS_ORDER], columns=[f"Pred_{l}" for l in LABELS_ORDER])
print(df_cm)


Starting Final Evaluation (K=2, Strict Token Logic, 2-Pass Optimization)
Params: Alpha=0.55, Threshold=-1.0
Metric: Words/second


100%|██████████| 10799/10799 [1:58:54<00:00,  1.51it/s]



[Final Results | K=2 Strict 2-Pass Optimized]
Inference Speed: 25.83 words/s
Total Processed Words: 184278
Total Execution Time: 7134.78s
------------------------------------------------------------
              precision    recall  f1-score   support

           O      0.985     0.973     0.979    160196
       COMMA      0.731     0.860     0.790     13017
      PERIOD      0.953     0.912     0.932     10153
       QMARK      0.855     0.888     0.871       912

    accuracy                          0.962    184278
   macro avg      0.881     0.908     0.893    184278
weighted avg      0.965     0.962     0.963    184278


Confusion Matrix
             Pred_O  Pred_COMMA  Pred_PERIOD  Pred_QMARK
True_O       155940        4048          189          19
True_COMMA     1608       11192          205          12
True_PERIOD     724          67         9256         106
True_QMARK       33           6           63         810
